# AlignTune: three-stage Hindi adaptation

Tokenizer extension → Hindi embedding-only continued pretraining → Hindi SFT with LoRA. Stage 3 starts from the merged Stage 2 checkpoint.

In [ ]:
!pip -q install transformers datasets accelerate peft trl sentencepiece
import os, subprocess
REPO_DIR='/content/aligntune'; BRANCH='feature/selective-embedding-training'
if not os.path.exists(os.path.join(REPO_DIR,'pyproject.toml')):
    subprocess.run(['git','clone','-b',BRANCH,'https://github.com/BhavyaGoyal777/aligntune.git',REPO_DIR],check=True)
!pip -q install -e /content/aligntune --no-deps

In [ ]:
import json, random, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from aligntune import create_sft_trainer, create_tokenization_trainer
MODEL_ID='LiquidAI/LFM2-350M'
TOKEN_DATASET,TOKEN_CONFIG='HuggingFaceFW/fineweb-2','hin_Deva'
SFT_DATASET,SFT_SPLIT='smangrul/hindi_instruct_v1','train'
TRAIN_DOCS,EVAL_DOCS,SFT_DOCS=1024,256,512
NEW_TOKENS=512; MAX_LENGTH=512; EMBEDDING_STEPS=200; SFT_STEPS=200; BATCH_SIZE=2; SEED=42
random.seed(SEED); torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# Hindi raw text for tokenizer extension and embedding-only continued pretraining.
stream=load_dataset(TOKEN_DATASET,TOKEN_CONFIG,split='train',streaming=True)
rows=[x for x in stream.take(TRAIN_DOCS+EVAL_DOCS) if x.get('text')]
assert len(rows)>=TRAIN_DOCS+EVAL_DOCS
train_path,eval_path='/content/fineweb_hin_train.jsonl','/content/fineweb_hin_eval.jsonl'
def write_jsonl(path,items):
    with open(path,'w',encoding='utf-8') as f:
        for item in items: f.write(json.dumps(item,ensure_ascii=False)+'\n')
write_jsonl(train_path,[{'text':x['text'],'language':'hi'} for x in rows[:TRAIN_DOCS]])
write_jsonl(eval_path,[{'text':x['text'],'language':'hi'} for x in rows[TRAIN_DOCS:]])
# Hindi instruction conversations for Stage 3 SFT.
sft_stream=load_dataset(SFT_DATASET,split=SFT_SPLIT,streaming=True)
sft_rows=[x for x in sft_stream.take(SFT_DOCS) if x.get('messages')]
assert len(sft_rows)>=SFT_DOCS
sft_path='/content/hindi_sft.jsonl'; write_jsonl(sft_path,[{'messages':x['messages']} for x in sft_rows])
print({'embedding_docs':TRAIN_DOCS,'eval_docs':EVAL_DOCS,'hindi_sft_conversations':len(sft_rows)})

In [ ]:
# STAGE 1: continued-BPE tokenizer extension through AlignTune.
tokenizer_output='/content/aligntune_tokenizer'
tok=create_tokenization_trainer(base_model=MODEL_ID,target_languages=['hi'],dataset_name=train_path,output_dir=tokenizer_output,num_new_tokens=NEW_TOKENS,extension_method='continued_bpe',text_column='text',max_samples=TRAIN_DOCS,seed=SEED)
tok_result=tok.train(); extended_tokenizer_path=tokenizer_output
base_tokenizer=AutoTokenizer.from_pretrained(MODEL_ID); extended_tokenizer=AutoTokenizer.from_pretrained(extended_tokenizer_path)
print({'base_vocab':len(base_tokenizer),'expanded_vocab':len(extended_tokenizer),'result':tok_result})

In [ ]:
def make_trainer(model_name,tokenizer_path,dataset_name,task_type,output_dir,train_embeddings,use_peft,max_steps):
    return create_sft_trainer(model_name=model_name,tokenizer_name_or_path=tokenizer_path,dataset_name=dataset_name,backend='trl',output_dir=output_dir,task_type=task_type,dataset_text_field='text',text_column='text',max_samples=TRAIN_DOCS if task_type=='pretraining' else SFT_DOCS,max_seq_length=MAX_LENGTH,batch_size=BATCH_SIZE,max_steps=max_steps,learning_rate=2e-3 if train_embeddings else 2e-4,weight_decay=0.0,embedding_init_method='mean_of_constituents',train_embeddings=train_embeddings,use_peft=use_peft,lora_r=16,lora_alpha=32,lora_dropout=0.05,seed=SEED)
# STAGE 2: Hindi embedding-only continued pretraining; only new rows train.
stage2_dir='/content/stage2_hindi_embedding_only'
stage2=make_trainer(MODEL_ID,extended_tokenizer_path,train_path,'pretraining',stage2_dir,True,False,EMBEDDING_STEPS)
stage2_result=stage2.train()
stage2_checkpoint='/content/stage2_hindi_merged'
stage2_model=stage2.model.base_model.merge_and_unload(); stage2_model.save_pretrained(stage2_checkpoint); extended_tokenizer.save_pretrained(stage2_checkpoint)
# STAGE 3: Hindi supervised fine-tuning with LoRA, starting from Stage 2.
stage3_dir='/content/stage3_hindi_sft_lora'
stage3=make_trainer(stage2_checkpoint,stage2_checkpoint,sft_path,'sft',stage3_dir,False,True,SFT_STEPS)
stage3_result=stage3.train()
print({'stage2_embedding_only':stage2_result,'stage3_hindi_sft_lora':stage3_result})

In [ ]:
def fertility(tokenizer,texts):
    return sum(len(tokenizer.encode(t,add_special_tokens=False)) for t in texts)/max(sum(len(t.split()) for t in texts),1)
eval_texts=[json.loads(x)['text'] for x in open(eval_path,encoding='utf-8')]
print({'source_fertility':fertility(base_tokenizer,eval_texts),'expanded_fertility':fertility(extended_tokenizer,eval_texts)})
print('Stage 3 is Hindi SFT with LoRA, not raw FineWeb pretraining.')

Data: FineWeb2 `hin_Deva` is used for tokenizer learning and embedding-only continued pretraining. `smangrul/hindi_instruct_v1` is used for Stage 3 Hindi instruction SFT; it provides a `messages` field and a Hindi train split.

Stage 2 uses AlignTune `train_embeddings=True` and no external LoRA. Stage 3 uses `task_type='sft'` and `use_peft=True` for LoRA.